# 03 - Attributes vs skill, and UK maps

In [ ]:
import sys
from pathlib import Path
FUNCTIONS_PATH = Path("/home/mokr/Reservoir_Modelling/ML_Functions/")
if str(FUNCTIONS_PATH) not in sys.path:
    sys.path.append(str(FUNCTIONS_PATH))

import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.patches import Patch
from scipy import stats

from ML_Plots import plot_models_vs_attributes
from analysis_config import AVERAGE_MODEL_COLORS, ANALYSIS_CACHE_PATH

with open(ANALYSIS_CACHE_PATH, "rb") as f:
    cache = pickle.load(f)
avg_metrics_by_results    = cache["avg_metrics_by_results"]
merged_metrics_by_results = cache["merged_metrics_by_results"]

## Static attributes

In [ ]:
folder_path = "/perm/mokr/CamelsGB_V2/Catchment_Attributes/"
All_Static_Variables = pd.DataFrame()
# List CSV files directly in the main folder (excluding subfolders)
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

# Loop through the CSV files
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    # Read the CSV file and concatenate it with the merged_df
    static_variable_group = pd.read_csv(file_path)
    All_Static_Variables = pd.concat(
        [All_Static_Variables, static_variable_group], ignore_index=True
    )

# The CSVs have different columns, so concatenating leaves rows with no gauge_id
All_Static_Variables = All_Static_Variables.dropna(subset=["gauge_id"]).reset_index(drop=True)

## Attribute vs NSE (reservoir gauges)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ---------------------------------------------------------
# 1. Clean and prepare reservoir_static
# ---------------------------------------------------------
target_gauges = avg_metrics_by_results['no_flow_reservoir'][
    'Reservoir_Embedded_LSTM'
].index
target_gauges_float = target_gauges.astype(float)

target_columns = [
    'gauge_id',
    'reservoir_contributing_area',
    'reservoir_normalised_upstream_capacity',
]
data_columns = [col for col in target_columns if col != 'gauge_id']

reservoir_static = All_Static_Variables.copy()
mask = reservoir_static['gauge_id'].isin(target_gauges_float)

reservoir_static = (
    reservoir_static.loc[mask, target_columns]
    .dropna(subset=data_columns, how='all')
    .drop_duplicates(subset=['gauge_id'], keep='first')
)

# Convert gauge_id to string to match the model results index
reservoir_static['gauge_id'] = (
    reservoir_static['gauge_id'].astype(int).astype(str)
)
reservoir_static = reservoir_static.set_index('gauge_id')

# ---------------------------------------------------------
# 2. Extract NSE scores and build the joint DataFrame
# ---------------------------------------------------------
models_dict = avg_metrics_by_results['no_flow_reservoir']

# Extract NSE Series for all models
nse_data = {
    f'NSE_{model_name}': model_df['NSE']
    for model_name, model_df in models_dict.items()
}
nse_df = pd.DataFrame(nse_data)

# Join static attributes with NSE metrics
joint_df = reservoir_static.join(nse_df, how='inner')

# ---------------------------------------------------------
# 3. Compute Delta NSE against the Non_Reservoir_LSTM baseline
# ---------------------------------------------------------
baseline_col = 'NSE_NonReservoir_Embedded_LSTM'

delta_cols = []
if baseline_col in joint_df.columns:
    for model_name in models_dict.keys():
        col_name = f'NSE_{model_name}'
        if col_name != baseline_col:
            delta_col = f'Delta_NSE_{model_name}'
            joint_df[delta_col] = joint_df[col_name] - joint_df[baseline_col]
            delta_cols.append(delta_col)

In [ ]:
# ---------------------------------------------------------
# 1. Plot Raw NSE for all models
# ---------------------------------------------------------
nse_cols = [c for c in joint_df.columns if c.startswith('NSE_')]
plot_models_vs_attributes(
    df=joint_df,
    attributes=data_columns,
    target_metric_cols=nse_cols,
    title_prefix="NSE",
    ylabel="NSE Score"
)

# ---------------------------------------------------------
# 2. Plot Delta NSE for all models (vs Baseline)
# ---------------------------------------------------------

if delta_cols:
    plot_models_vs_attributes(
        df=joint_df,
        attributes=data_columns,
        target_metric_cols=delta_cols,
        title_prefix="Δ NSE (vs Baseline)",
        ylabel="Δ NSE"
    )

## Maps

In [ ]:
import geopandas as gpd
import os

uk_boundary = gpd.read_file(
    "/home/mokr/Camels_GB/GB_Shape_Files/CTRY_DEC_2024_UK_BFC.shp"
)

In [ ]:
# Pick model and metric to colour by
MODEL = "Hydra Specialist"
METRIC = "KGE"

# ['Specialist', 'Full LSTM', 'Hydra_Specialist', 'Hydra: Main']

# Build a scores Series with string gauge_id index
scores = merged_metrics_by_results['all_gauges'][MODEL][METRIC].copy()
scores.index = scores.index.astype(int).astype(str)

# Filter static to merged gauges and attach scores
filtered_static = All_Static_Variables.copy()
filtered_static['gauge_id'] = filtered_static['gauge_id'].astype(int).astype(str)
filtered_static = filtered_static[filtered_static['gauge_id'].isin(scores.index)]
filtered_static[METRIC] = filtered_static['gauge_id'].map(scores)

# Build GeoDataFrame
gdf_gauges = gpd.GeoDataFrame(
    filtered_static,
    geometry=gpd.points_from_xy(filtered_static["gauge_lon"], filtered_static["gauge_lat"]),
    crs="EPSG:4326",
)
if uk_boundary.crs != gdf_gauges.crs:
    gdf_gauges = gdf_gauges.to_crs(uk_boundary.crs)

# Plot
fig, ax = plt.subplots(figsize=(12, 14))
uk_boundary.plot(ax=ax, color="lightgray", edgecolor="darkgray", linewidth=1.5)

scatter = gdf_gauges.plot(
    ax=ax,
    column=METRIC,
    cmap="RdYlGn",
    vmin=-0.5, vmax=1.0,
    markersize=45,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.5,
    legend=False,
)

# Colorbar
sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(vmin=-0.5, vmax=1.0))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label(f"{MODEL} — {METRIC}", fontsize=13)
cbar.ax.tick_params(labelsize=11)

ax.set_title(f"Gauge {METRIC} scores — {MODEL}", fontsize=16, fontweight="bold", pad=20)
ax.set_xlabel("Easting", fontsize=12)
ax.set_ylabel("Northing", fontsize=12)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch
METRIC = "KGE"


models = list(merged_metrics_by_results['all_gauges'].keys())

# Build a DataFrame of scores: index=gauge_id, columns=models
scores_df = pd.DataFrame({
    model: merged_metrics_by_results['all_gauges'][model][METRIC]
    for model in models
})
scores_df.index = scores_df.index.astype(int).astype(str)

# Find best model per gauge
scores_df['best_model'] = scores_df[models].idxmax(axis=1)
scores_df['color'] = scores_df['best_model'].map(AVERAGE_MODEL_COLORS)

# Filter static and attach
filtered_static = All_Static_Variables.copy()
filtered_static['gauge_id'] = filtered_static['gauge_id'].astype(int).astype(str)
filtered_static = filtered_static[filtered_static['gauge_id'].isin(scores_df.index)]
filtered_static['best_model'] = filtered_static['gauge_id'].map(scores_df['best_model'])
filtered_static['color'] = filtered_static['gauge_id'].map(scores_df['color'])

# Build GeoDataFrame
gdf_gauges = gpd.GeoDataFrame(
    filtered_static,
    geometry=gpd.points_from_xy(filtered_static["gauge_lon"], filtered_static["gauge_lat"]),
    crs="EPSG:4326",
)
if uk_boundary.crs != gdf_gauges.crs:
    gdf_gauges = gdf_gauges.to_crs(uk_boundary.crs)

# Plot
fig, ax = plt.subplots(figsize=(12, 14))
uk_boundary.plot(ax=ax, color="lightgray", edgecolor="darkgray", linewidth=1.5)

gdf_gauges.plot(
    ax=ax,
    color=gdf_gauges['color'],
    markersize=45,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.5,
)

# Legend
legend_elements = [
    Patch(facecolor=AVERAGE_MODEL_COLORS[model], edgecolor="black", label=model)
    for model in models
]
ax.legend(
    handles=legend_elements,
    title=f"Best model ({METRIC})",
    loc="upper left",
    frameon=True,
    fancybox=True,
    shadow=True,
)

ax.set_title(f"Best performing model per gauge — {METRIC}", fontsize=16, fontweight="bold", pad=20)
ax.set_xlabel("Easting", fontsize=12)
ax.set_ylabel("Northing", fontsize=12)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print("\nBest model counts:")
print(scores_df['best_model'].value_counts())